# reidfm: build the dataset pool (local machine / Lightning / any box with the raw data)

Converts every raw dataset into `<out>/<key>/{manifest.csv, meta.json, images/}`, audits it
(leakage, OOD roles, cross-dataset duplicates), and packages it for upload as Kaggle Datasets.

Restricted datasets (MSMT17, LaST, DeepChange, Occluded-PoseTrack, ORBench) must be uploaded
as **private** Kaggle Datasets only.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO); sys.path.insert(0, str(REPO))

# ---- edit these ----
os.environ["REIDFM_RAW_ROOT"] = r"/path/to/raw/datasets"   # folders like Market-1501-v15.09.15, MSMT17_V1, MARS ...
OUT_ROOT = REPO / "pool"
POOL = "configs/datasets/pool_v1.yaml"
MATERIALIZE = "copy"     # copy for upload; "link" saves disk for local training
WORKERS = 16
# --------------------

In [ ]:
# Dry run: run the converters and print per-dataset counts without writing anything.
!python scripts/prepare_datasets.py --pool $POOL --out-root "$OUT_ROOT" --dry-run

In [ ]:
# Convert + audit. --skip-missing lets you build what you have now and add the rest later.
!python scripts/prepare_datasets.py --pool $POOL --out-root "$OUT_ROOT" --materialize $MATERIALIZE --workers $WORKERS --skip-missing --dedup-check

In [ ]:
# Package for Kaggle: one folder per Kaggle Dataset, each holding <key>/ sub-folders.
# find_manifest() looks one level below REIDFM_DATA_ROOT, so /kaggle/input/<slug>/<key>/manifest.csv works.
import shutil
GROUPS = {
    "reid-pool-image": ["market1501", "cuhk03np", "prw", "iust", "rpifield"],
    "reid-pool-image-restricted": ["msmt17", "last", "deepchange", "occposetrack", "orbenchrgb"],
    "reid-pool-video": ["mars", "lpw", "prid2011"],
    "reid-pool-mot": ["mot17", "mot20"],
}
UPLOAD = REPO / "kaggle_upload"
for slug, keys in GROUPS.items():
    for key in keys:
        src = OUT_ROOT / key
        if src.is_dir():
            dst = UPLOAD / slug / key
            if not dst.exists():
                shutil.copytree(src, dst)
    print(slug, sorted(p.name for p in (UPLOAD / slug).glob("*")) if (UPLOAD / slug).exists() else "-")

Upload each folder, e.g. with the Kaggle CLI:

```bash
kaggle datasets init -p kaggle_upload/reid-pool-image      # edit dataset-metadata.json (id, title)
kaggle datasets create -p kaggle_upload/reid-pool-image --dir-mode zip
```

Use `--public` only for datasets whose license allows redistribution (see `THIRD_PARTY_PROVENANCE.md`).